# GPU-Accelerated Geant4 Simulation of the ATLAS Tile Calorimeter

This notebook walks you through a compact end-to-end example of a **CUDA-enabled Geant4 + DD4hep + Celeritas** workflow. You will prepare a realistic slice of the ATLAS Tile Calorimeter geometry, build a minimal application, compare CPU and GPU execution, and inspect the resulting energy-deposit spectrum.

The material is written as a hands-on exercise for learners who are new to HEP software stacks, while still being useful for readers who already know Geant4 or CUDA and want a concise working example.

**Learning objectives**

* Prepare or verify the software environment needed for the notebook
* Fetch detector geometry and steering files from DD4hep/Celeritas examples
* Compile a minimal C++ application that offloads electromagnetic transport to the GPU
* Compare CPU and GPU timing for the same workflow
* Complete two short follow-up exercises using the provided scaffolding

## 0  Environment bootstrap (optional)
If you already have a suitable Python environment, you can skip this section. Otherwise, run the commented commands below in a terminal to create a Conda environment named `tilegpu` and start JupyterLab.

```bash
# conda create -n tilegpu -c conda-forge python=3.11 jupyterlab
# conda activate tilegpu
# jupyter lab
```

> The rest of the notebook assumes you are working inside an environment that can run Python, Jupyter, and the helper packages installed below.

## Official Celeritas Installation (Spack)

For a real installation, the recommended route is to install Celeritas and its dependencies with Spack. The commands below summarize the typical setup flow.

```bash
# Install Spack
git clone --depth=2 https://github.com/spack/spack.git
. spack/share/spack/setup-env.sh

# Set up CUDA (if you have a GPU)
spack external find cuda

# Set default configuration (replace cuda_arch=80 with your GPU architecture)
spack config add packages:all:variants:"cxxstd=17 +cuda cuda_arch=80"

# Install Celeritas
spack install celeritas
spack load celeritas
```

If you are using this notebook in a teaching environment, it is fine to treat these commands as reference material and rely on a pre-prepared installation instead.

See the [Celeritas documentation](https://github.com/celeritas-project/celeritas) for more details and integration steps.

In [ ]:
# Install lightweight Python deps (≈1 min on first run)
%pip install --quiet cupy-cuda12x uproot awkward matplotlib nbformat


### Why Celeritas + DD4hep?
* **Celeritas** accelerates Geant4 electromagnetic (e±/γ) transport on NVIDIA GPUs, yielding order-of-magnitude speed-ups for calorimeter studies.
* **DD4hep** supplies experiment-quality detector descriptions; the TileCal GDML used here is taken from the public ATLAS repository.


**Note:**

This notebook assumes that Celeritas and Geant4 are already installed and available in your environment, ideally through the Spack workflow shown above. The remaining cells focus on using that installation rather than building the full software stack from scratch.

See the [Celeritas documentation](https://github.com/celeritas-project/celeritas) for integration details.

## 1  Prepare ATLAS TileCal Geometry and Macro
The next cell downloads a small Tile Calorimeter GDML plus a steering macro that shoots **10 GeV electrons** along the beam axis.


In [ ]:
import urllib.request, pathlib, os

files = {
    "TileTB_2B1EB_nobeamline.gdml": "https://raw.githubusercontent.com/celeritas-project/atlas-tilecal-integration/main/TileTB_2B1EB_nobeamline.gdml",
    "TBrun_elec.mac": "https://raw.githubusercontent.com/celeritas-project/atlas-tilecal-integration/main/TBrun_elec.mac"
}
for fname, url in files.items():
    if not pathlib.Path(fname).exists():
        print(f"Downloading {fname} …")
        urllib.request.urlretrieve(url, fname)
    else:
        print(f"{fname} already present")
print("Geometry & macro ready.")


## 2  Minimal Geant4 + Celeritas Off-load Application
Below we write **<60 lines** of C++ that
1. Build a multithreaded Geant4 run manager
2. Load the TileCal geometry with DD4hep
3. Insert *Celeritas* as the parallel GPU transport
4. Execute the macro to shoot particles

> **Tip:** Compiling inside Jupyter uses the system compiler via shell commands starting with `!`.


In [ ]:
%%writefile tile_gpu.cc
#include <G4RunManagerFactory.hh>
#include <G4UImanager.hh>
#include <FTFP_BERT.hh>
#include <DDG4/Geant4DetectorConstruction.h>
#include <celeritas/TrackingManagerConstructor.hh>

int main()
{
    auto rm = G4RunManagerFactory::CreateRunManager(G4RunManagerType::MT);
    rm->SetUserInitialization(new dd4hep::sim::Geant4DetectorConstruction("TileTB_2B1EB_nobeamline.gdml"));

    auto* phys = new FTFP_BERT;
    phys->RegisterPhysics(new celeritas::TrackingManagerConstructor);
    rm->SetUserInitialization(phys);

    rm->Initialize();

    auto* ui = G4UImanager::GetUIpointer();
    ui->ApplyCommand("/tracking/verbose 0");
    ui->ApplyCommand("/control/execute TBrun_elec.mac");

    delete rm;
    return 0;
}


In [ ]:
# Compile the application
!g++ tile_gpu.cc -std=c++20 -O3 $(geant4-config --cflags --libs) -I$CELER_DIR/include -L$CELER_DIR/lib -lCeleritas -o tile_gpu
print("Compilation finished → executable ./tile_gpu")


## 3  Benchmark CPU vs GPU
In this section you will run the same application twice: once with Celeritas forced into CPU-only mode and once with GPU transport enabled. The helper function below switches between the two modes using the `CELER_DISABLE_DEVICE` environment variable.

As you read the output, focus on relative timing rather than absolute numbers. The exact runtime will depend strongly on the machine and software environment.

In [ ]:
import subprocess, time, os, re

def run_tile(use_gpu: bool = True):
    env = os.environ.copy()
    env["CELER_DISABLE_DEVICE"] = "0" if use_gpu else "1"
    start = time.time()
    output = subprocess.check_output(["./tile_gpu"], text=True, env=env)
    dt = time.time() - start
    m = re.search(r"Run summary: *(\d+) events", output)
    nev = int(m.group(1)) if m else 1000
    return dt, nev

cpu_t, nevt = run_tile(False)
gpu_t, _ = run_tile(True)
print(
    f"Simulated {nevt} events\n"
    f"CPU time : {cpu_t:.2f} s\n"
    f"GPU time : {gpu_t:.2f} s\n"
    f"Speed-up : x{cpu_t / gpu_t:.1f}"
)

## 4  Visualise Energy Deposits
After the run completes, the Geant4 scorer writes ROOT output. The next cell opens that file with `uproot` and makes a simple log-scale histogram so you can inspect the energy-deposit distribution at the hit level.

If the cell raises a file-not-found error, go back and run the simulation first.

In [ ]:
import matplotlib.pyplot as plt, awkward as ak, uproot, pathlib

root_files = list(pathlib.Path('.').glob('*.root'))
if not root_files:
    raise FileNotFoundError("No ROOT output file found. Run the simulation cell first.")

root_file = root_files[0]
print('Reading', root_file)
file = uproot.open(str(root_file))
edep = file["TileCellHits/Edep"].array()

plt.figure(figsize=(6, 4))
plt.hist(edep.to_numpy(), bins=120, log=True, histtype='step')
plt.xlabel('Energy deposit per hit [MeV]')
plt.ylabel('Counts / bin')
plt.title('10 GeV e- energy deposits (GPU offload)')
plt.tight_layout()
plt.show()

## 5  Exercises (≈15 min)

These exercises are meant to extend the notebook without requiring a completely new setup. Treat them as guided experiments: make one change, rerun the relevant steps, and compare what changes in both the physics output and the timing.

### Exercise 1 – Hadron shower
Modify **`TBrun_elec.mac`** so the beam uses **30 GeV π⁺** instead of electrons, then repeat the timing study. Compare the result with the electron case and note that the GPU advantage is smaller because a larger fraction of the hadronic cascade remains on the CPU.

### Exercise 2 – Absorber thickness scan
1. Use the (mock) macro command `/dd/geometry/scaleAbsorber X cm` where *X* runs from 1 to 5 cm.
2. Loop over thickness values and record both (a) total visible energy and (b) GPU speed-up.
3. Plot the two trends together using dual y-axes.

A short scaffolding cell is provided below so you can focus on the experiment logic rather than notebook boilerplate.

In [ ]:
# —— Solution scaffolding ——
import json, numpy as np

def run_with_macro(macro_text):
    macro_path = 'tmp.mac'
    with open(macro_path,'w') as f:
        f.write(macro_text)
    env = os.environ.copy()
    env["CELER_DISABLE_DEVICE"] = "1"
    subprocess.check_output(["./tile_gpu"], env=env, text=True)
    os.remove(macro_path)
    # add return values extraction if needed

# Students fill in details based on above helpers


## 6  Clean-up
Use the following command if you want to remove generated binaries, temporary build products, and large output files after finishing the exercise.

You can safely skip this step if you want to keep the artifacts for later inspection.

In [ ]:
!rm -rf geant4_celeritas_cuda12.tgz geant4_cuda *.root tile_gpu tile_gpu.cc
